In [1]:
# ==========================================
# CELL 1: SETUP & IMPORTS
# ==========================================
import os
import torch
import numpy as np
import sentencepiece as spm
from datasets import load_dataset
from transformers import T5Config, T5ForConditionalGeneration, PreTrainedTokenizerFast

# Suppress some Hugging Face warnings for a cleaner output
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Verify hardware
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU Name: Quadro RTX 6000


In [7]:
# ==========================================
# CELL 2: LOAD & SAMPLE PRE-TRAINING DATA
# ==========================================
print("Loading CodeSearchNet (Java subset)...")

csn = load_dataset("code_search_net", "java", trust_remote_code=True)

# Sample 50,000 methods with seed 42 for reproducibility
methods = csn["train"].shuffle(seed=42).select(range(50000))

# Extract the whole_func_string 
def extract_body(example):
    return {"text": example["whole_func_string"]}

print("Extracting method bodies...")
pretrain_dataset = methods.map(extract_body, remove_columns=methods.column_names)

# Save raw text to a file for the SentencePiece trainer
corpus_file = "pretrain_corpus.txt"
print(f"Saving corpus to {corpus_file} for tokenizer training...")
with open(corpus_file, "w", encoding="utf-8") as f:
    for item in pretrain_dataset:
        # Replacing newlines with spaces ensures the SP trainer reads it cleanly, 
        # though SP can handle newlines, this avoids accidental multi-line truncation
        f.write(item["text"].replace('\n', ' ') + "\n")

print(f"Successfully prepared {len(pretrain_dataset)} methods for pre-training.")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'code_search_net' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading CodeSearchNet (Java subset)...


Extracting method bodies...
Saving corpus to pretrain_corpus.txt for tokenizer training...
Successfully prepared 50000 methods for pre-training.


In [8]:
# ==========================================
# CELL 3 & 4: TOKENIZER TRAINING (FAST METHOD)
# ==========================================
import sentencepiece as spm
from tokenizers import Tokenizer
from tokenizers.models import Unigram
from tokenizers.pre_tokenizers import Metaspace
from tokenizers.decoders import Metaspace as MetaspaceDecoder
from transformers import PreTrainedTokenizerFast

vocab_size = 16384
model_prefix = "sp_code"

# 1. Train SentencePiece Unigram Tokenizer
print("1. Training SentencePiece model...")
sentinel_tokens = [f"<extra_id_{i}>" for i in range(100)]
spm.SentencePieceTrainer.train(
    input=corpus_file,
    model_prefix=model_prefix,
    vocab_size=vocab_size,
    model_type="unigram",
    user_defined_symbols=",".join(sentinel_tokens),
    pad_id=0,
    eos_id=1,
    unk_id=2,
    bos_id=-1,
    hard_vocab_limit=False,
    character_coverage=1.0
)

# 2. Convert to HuggingFace Fast Tokenizer
print("2. Converting to Hugging Face PreTrainedTokenizerFast...")
sp = spm.SentencePieceProcessor()
sp.Load(f"{model_prefix}.model")

# Extract vocab as list of (piece, score) tuples
vocab = [(sp.IdToPiece(i), sp.GetScore(i)) for i in range(sp.GetPieceSize())]

# Build HF tokenizer object
tokenizer_obj = Tokenizer(Unigram(vocab, unk_id=2))
tokenizer_obj.pre_tokenizer = Metaspace()
tokenizer_obj.decoder = MetaspaceDecoder()

# Wrap as a PreTrainedTokenizerFast
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer_obj,
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
    additional_special_tokens=sentinel_tokens,
)

# Save for later
tokenizer.save_pretrained("./java_tokenizer")
print(f"Tokenizer saved! Vocab size: {len(tokenizer)}")

# 3. Diagnostic Check
print("\n3. DIAGNOSTIC CHECK (Confirming valid token IDs):")
sample_code = "public static void main(String[] args)"
print(f"Tokens: {tokenizer.tokenize(sample_code)}")
print(f"Token IDs: {tokenizer(sample_code)['input_ids']}")

# 4. Filter pre-training dataset
print("\n4. Filtering Dataset...")
def filter_by_length(example):
    token_ids = tokenizer(example["text"], truncation=False)["input_ids"]
    return 10 <= len(token_ids) <= 512

filtered_pretrain_dataset = pretrain_dataset.filter(filter_by_length, num_proc=4)
print(f"Methods remaining: {len(filtered_pretrain_dataset)} out of {len(pretrain_dataset)}")

1. Training SentencePiece model...
2. Converting to Hugging Face PreTrainedTokenizerFast...
Tokenizer saved! Vocab size: 16384

3. DIAGNOSTIC CHECK (Confirming valid token IDs):
Tokens: ['▁public', '▁static', '▁void', '▁main', '(', 'String', '[]', '▁args', ')']
Token IDs: [121, 154, 140, 1961, 104, 131, 164, 648, 107]

4. Filtering Dataset...
Methods remaining: 43863 out of 50000


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: pretrain_corpus.txt
  input_format: 
  model_prefix: sp_code
  model_type: UNIGRAM
  vocab_size: 16384
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: <extra_id_0>
  user_defined_symbols: <extra_id_1>
  user_defined_symbols: <extra_id_2>
  user_defined_symbols: <extra_id_3>
  user_defined_symbols: <extra_id_4>
  user_defined_symbols: <extra_id_5>
  user_defined_symbols: <extra_id_6>
  user_defined_symbols: <extra_id_7>
  user_defined_symbols: <extra_id_8>
  user_defined_symbols:

In [9]:
# ==========================================
# CELL 5: MANUAL T5-SMALL ARCHITECTURE INITIALIZATION
# ==========================================
from transformers import T5Config, T5ForConditionalGeneration

print("Defining T5-small configuration manually...")

# Configuration corresponding exactly to a T5-small architecture
t5_config = T5Config(
    vocab_size=len(tokenizer),
    decoder_start_token_id=0,
    eos_token_id=1,
    bos_token_id=0,
    pad_token_id=0,
    d_model=512,
    d_ff=2048,
    d_kv=64,
    num_heads=8,
    num_layers=6,
    num_decoder_layers=6
)

print("Initializing the model from scratch (uninitialized weights)...")
model = T5ForConditionalGeneration(config=t5_config)

# Resize token embeddings to precisely match our tokenizer's vocabulary
model.resize_token_embeddings(len(tokenizer))

# Move the model to our RTX 6000
model = model.to(device)

# Verify parameters (should be around ~60M)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model successfully initialized on {device}!")
print(f"Total Parameters: {total_params:,}")

Defining T5-small configuration manually...
Initializing the model from scratch (uninitialized weights)...
Model successfully initialized on cuda!
Total Parameters: 52,445,696


In [14]:
# ==========================================
# CELL 6: SPAN CORRUPTION & DATA PREPARATION
# ==========================================
import random
import numpy as np

print("Tokenizing and applying span corruption (15% rate)...")

def prepare_span_corruption(examples):
    # First, tokenize the batch
    tokenized = tokenizer(examples["text"], truncation=True, max_length=512, padding=False)
    
    inputs_batch = []
    labels_batch = []
    
    for input_ids in tokenized["input_ids"]:
        # 1. Determine which tokens to corrupt (15% probability)
        # We avoid corrupting special tokens (pad, eos, unk) if they exist
        seq_len = len(input_ids)
        mask = np.random.rand(seq_len) < 0.15
        
        # 2. Build the input sequence (with sentinels) and label sequence (the corrupted tokens)
        corrupted_inputs = []
        target_labels = []
        
        sentinel_idx = 0
        is_active_span = False
        
        for i, token in enumerate(input_ids):
            # Skip special tokens in masking
            if token in [tokenizer.pad_token_id, tokenizer.eos_token_id, tokenizer.unk_token_id]:
                corrupted_inputs.append(token)
                is_active_span = False
                continue
                
            if mask[i]:
                # It's a corrupted token
                if not is_active_span:
                    # Start a new span
                    sentinel_token = tokenizer.convert_tokens_to_ids(f"<extra_id_{sentinel_idx}>")
                    corrupted_inputs.append(sentinel_token)
                    target_labels.append(sentinel_token)
                    sentinel_idx += 1
                    is_active_span = True
                
                target_labels.append(token)
            else:
                # Normal token
                corrupted_inputs.append(token)
                is_active_span = False
                
        # Close out sequences with EOS token
        if corrupted_inputs[-1] != tokenizer.eos_token_id:
            corrupted_inputs.append(tokenizer.eos_token_id)
        target_labels.append(tokenizer.eos_token_id)
        
        inputs_batch.append(corrupted_inputs)
        labels_batch.append(target_labels)
        
    return {"input_ids": inputs_batch, "labels": labels_batch}

# Apply the mapping
# We remove the 'text' column because the model only expects input_ids and labels
pretrain_dataset_tokenized = filtered_pretrain_dataset.map(
    prepare_span_corruption, 
    batched=True, 
    remove_columns=["text"],
    num_proc=4
)

print("Dataset prepared for span corruption!")
print(f"Sample Input length: {len(pretrain_dataset_tokenized[0]['input_ids'])}")
print(f"Sample Label length: {len(pretrain_dataset_tokenized[0]['labels'])}")

Tokenizing and applying span corruption (15% rate)...
Dataset prepared for span corruption!
Sample Input length: 125
Sample Label length: 39


In [15]:
# ==========================================
# CELL 7: PRE-TRAINING LOOP
# ==========================================
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

print("Setting up TrainingArguments for Pre-training...")

# Configure training arguments exactly as requested:
# 3 epochs, flat training, logging per epoch to show decreasing loss
pretrain_args = TrainingArguments(
    output_dir="./t5_pretrained_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    learning_rate=5e-4,
    logging_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    fp16=True,
    report_to="none"
)

# A standard data collator will handle dynamically padding the inputs and labels to the max length in the batch
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

pretrain_trainer = Trainer(
    model=model,
    args=pretrain_args,
    train_dataset=pretrain_dataset_tokenized,
    data_collator=data_collator,
)

print("Starting Pre-training... (This will take some time)")
pretrain_results = pretrain_trainer.train()

print("Pre-training Complete!")
print("Saving the final pre-trained model...")
pretrain_trainer.save_model("./t5_final_pretrained")
tokenizer.save_pretrained("./t5_final_pretrained")

# Print out the logged losses per epoch to confirm they decreased
print("\n--- Training Loss History ---")
for log in pretrain_trainer.state.log_history:
    if "loss" in log and "epoch" in log:
        print(f"Epoch {log['epoch']:.1f} | Loss: {log['loss']:.4f}")

[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Setting up TrainingArguments for Pre-training...
Starting Pre-training... (This will take some time)


Step,Training Loss
1371,3.319653
2742,2.913215
4113,2.818889


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Pre-training Complete!
Saving the final pre-trained model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Training Loss History ---
Epoch 1.0 | Loss: 3.3197
Epoch 2.0 | Loss: 2.9132
Epoch 3.0 | Loss: 2.8189


In [16]:
# ==========================================
# CELL 8: FINE-TUNING DATA PREPARATION
# ==========================================
print("Loading CodeXGLUE Code Refinement Dataset (Medium)...")
finetune_dataset = load_dataset("google/code_x_glue_cc_code_refinement", name="medium")

print("Tokenizing buggy (input) and fixed (label) pairs...")

def prepare_finetune_data(examples):
    # Tokenize the buggy code (inputs)
    model_inputs = tokenizer(examples["buggy"], max_length=512, truncation=True, padding=False)
    
    # Tokenize the fixed code (labels)
    labels = tokenizer(examples["fixed"], max_length=512, truncation=True, padding=False)
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply tokenization across train, validation, and test splits
tokenized_finetune_datasets = finetune_dataset.map(
    prepare_finetune_data,
    batched=True,
    remove_columns=["buggy", "fixed", "id"], # Remove original columns
    num_proc=4
)

print("Fine-tuning data prepared successfully!")
print(f"Train samples: {len(tokenized_finetune_datasets['train'])}")
print(f"Validation samples: {len(tokenized_finetune_datasets['validation'])}")

Loading CodeXGLUE Code Refinement Dataset (Medium)...
Tokenizing buggy (input) and fixed (label) pairs...
Fine-tuning data prepared successfully!
Train samples: 52364
Validation samples: 6546


In [27]:
# ==========================================
# CELL 9: PIPELINE A (WITH PRE-TRAINING)
# ==========================================
import gc
import torch
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq, T5ForConditionalGeneration

# Clear VRAM before starting
gc.collect()
torch.cuda.empty_cache()

print("Loading PRE-TRAINED model for Pipeline A...")
# Load the model you successfully pre-trained in Cell 7
model_A = T5ForConditionalGeneration.from_pretrained("./t5_final_pretrained").to(device)

# Standard Fine-tuning Arguments (fp16 is DISABLED for stability on RTX 6000)
finetune_args = TrainingArguments(
    output_dir="./t5_finetuned_pipeline_A",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=1,
    fp16=False,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model_A, padding=True)

trainer_A = Trainer(
    model=model_A,
    args=finetune_args,
    train_dataset=tokenized_finetune_datasets["train"],
    eval_dataset=tokenized_finetune_datasets["validation"],
    data_collator=data_collator,
)

print("Starting Fine-tuning for Pipeline A (Pre-trained)...")
trainer_A.train()

print("Pipeline A Complete! Saving best model...")
trainer_A.save_model("./t5_best_pipeline_A")

Loading PRE-TRAINED model for Pipeline A...


Loading weights: 100%|██████████| 131/131 [00:00<00:00, 499.82it/s]


Starting Fine-tuning for Pipeline A (Pre-trained)...


Epoch,Training Loss,Validation Loss
1,1.000984,0.888428
2,0.841104,0.757496
3,0.767302,0.695093
4,0.717340,0.649002
5,0.690121,0.629748


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.04s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Pipeline A Complete! Saving best model...


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.88s/it]


In [17]:
# ==========================================
# CELL 10: PIPELINE B (NO PRE-TRAINING)
# ==========================================
import gc
import torch
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq, T5ForConditionalGeneration

# Clear VRAM from Pipeline A
del model_A, trainer_A
gc.collect()
torch.cuda.empty_cache()

print("Initializing FRESH model for Pipeline B...")
# We use the exact same t5_config we manually defined in Cell 5
model_B = T5ForConditionalGeneration(config=t5_config)
model_B.resize_token_embeddings(len(tokenizer))
model_B = model_B.to(device)

# Using the EXACT SAME hyperparameters as Pipeline A
finetune_args_B = TrainingArguments(
    output_dir="./t5_finetuned_pipeline_B",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=1,
    fp16=False,
    report_to="none"
)

trainer_B = Trainer(
    model=model_B,
    args=finetune_args_B,
    train_dataset=tokenized_finetune_datasets["train"],
    eval_dataset=tokenized_finetune_datasets["validation"],
    data_collator=data_collator,
)

print("Starting Fine-tuning for Pipeline B (From Scratch)...")
trainer_B.train()

print("Pipeline B Complete! Saving best model...")
trainer_B.save_model("./t5_best_pipeline_B")

Initializing FRESH model for Pipeline B...


[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Starting Fine-tuning for Pipeline B (From Scratch)...


Epoch,Training Loss,Validation Loss
1,0.510717,0.449132
2,0.415013,0.371784
3,0.371692,0.332856
4,0.334535,0.310770
5,0.313106,0.294025


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Pipeline B Complete! Saving best model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [23]:
# ==========================================
# CELL 11: EVALUATION SETUP
# ==========================================
import torch
import logging
from tqdm.auto import tqdm
from codebleu import calc_codebleu
from transformers import T5ForConditionalGeneration

print("Loading raw test dataset for evaluation...")
test_dataset = finetune_dataset["test"]
print(f"Test Set Size: {len(test_dataset)} samples")

def evaluate_model(model_dir, dataset, batch_size=32):
    print(f"\nLoading model from {model_dir} for evaluation...")
    eval_model = T5ForConditionalGeneration.from_pretrained(model_dir).to(device)
    eval_model.eval()
    
    predictions = []
    references = dataset["fixed"]
    
    # Generation loop
    for i in tqdm(range(0, len(dataset), batch_size), desc="Generating Fixes"):
        batch_inputs = dataset["buggy"][i : i + batch_size]
        
        inputs = tokenizer(
            batch_inputs, 
            return_tensors="pt", 
            padding=True, 
            truncation=True, 
            max_length=512
        ).to(device)
        
        with torch.no_grad():
            outputs = eval_model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"], # <--- THE FIX (Prevents OOM)
                max_new_tokens=256,                      # <--- THE FIX (Bounds memory)
                num_beams=3,
                early_stopping=True
            )
            
        decoded_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        predictions.extend(decoded_preds)
        
    print("Calculating Metrics...")
    
    # 1. Exact Match
    exact_matches = sum(1 for p, r in zip(predictions, references) if p.strip() == r.strip())
    em_score = (exact_matches / len(references)) * 100
    
    # 2. CodeBLEU 
    refs_for_bleu = [[r] for r in references]
    logging.getLogger().setLevel(logging.ERROR)
    
    try:
        cb_results = calc_codebleu(references=refs_for_bleu, predictions=predictions, lang="java")
        codebleu_score = cb_results["codebleu"] * 100
    except Exception as e:
        print(f"CodeBLEU Error: {e}")
        codebleu_score = 0.0
    
    # Free up VRAM
    del eval_model
    torch.cuda.empty_cache()
    
    return {
        "Exact Match (%)": round(em_score, 4),
        "CodeBLEU": round(codebleu_score, 4)
    }

print("Evaluation functions ready!")

Loading raw test dataset for evaluation...
Test Set Size: 6545 samples
Evaluation functions ready!


In [24]:
# ==========================================
# CELL 12: RUN EVALUATION ON PIPELINES A & B
# ==========================================

print("Evaluating Pipeline A (Pre-trained)...")
results_A = evaluate_model("./t5_best_pipeline_A", test_dataset, batch_size=32)

print("\nEvaluating Pipeline B (From Scratch)...")
results_B = evaluate_model("./t5_best_pipeline_B", test_dataset, batch_size=32)

print("\n" + "="*50)
print("FINAL EVALUATION RESULTS (FULL TEST SET)")
print("="*50)
print(f"Pipeline A (Pre-trained):")
print(f"  -> Exact Match: {results_A['Exact Match (%)']}%")
print(f"  -> CodeBLEU:    {results_A['CodeBLEU']}")
print("-" * 50)
print(f"Pipeline B (From Scratch):")
print(f"  -> Exact Match: {results_B['Exact Match (%)']}%")
print(f"  -> CodeBLEU:    {results_B['CodeBLEU']}")
print("="*50)

Evaluating Pipeline A (Pre-trained)...

Loading model from ./t5_best_pipeline_A for evaluation...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Generating Fixes:   0%|          | 0/205 [00:00<?, ?it/s]

Calculating Metrics...

Evaluating Pipeline B (From Scratch)...

Loading model from ./t5_best_pipeline_B for evaluation...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Generating Fixes:   0%|          | 0/205 [00:00<?, ?it/s]

Calculating Metrics...

FINAL EVALUATION RESULTS (FULL TEST SET)
Pipeline A (Pre-trained):
  -> Exact Match: 0.0%
  -> CodeBLEU:    30.726
--------------------------------------------------
Pipeline B (From Scratch):
  -> Exact Match: 0.0458%
  -> CodeBLEU:    51.9902


In [30]:
# ==========================================
# CELL 13: RAG SETUP (CODEBERT + FAISS)
# ==========================================
import os
import torch
import json
import logging
import numpy as np
import faiss
from tqdm.auto import tqdm
from codebleu import calc_codebleu
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

# Suppress logging for cleaner metric calculation
logging.getLogger().setLevel(logging.ERROR)

print("Loading CodeBERT for semantic retrieval...")
retriever_tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
retriever_model = AutoModel.from_pretrained("microsoft/codebert-base").to(device)
retriever_model.eval()

def encode_code(code_list, batch_size=32, desc="Encoding"):
    """Encodes code snippets using CodeBERT mean-pooling."""
    embeddings = []
    for i in tqdm(range(0, len(code_list), batch_size), desc=desc):
        batch = code_list[i : i + batch_size]
        inputs = retriever_tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = retriever_model(**inputs)
            # Mean pooling over the last hidden state for a single vector representation
            batch_embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            embeddings.append(batch_embeddings)
    return np.vstack(embeddings)

print("Encoding training knowledge base (buggy methods)...")
# Knowledge base: bug-fixing training data (buggy -> fixed pairs)
train_buggy = finetune_dataset["train"]["buggy"]
train_fixed = finetune_dataset["train"]["fixed"]
kb_embeddings = encode_code(train_buggy, desc="Indexing KB")

# Build FAISS index for fast L2 similarity search
dimension = kb_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(kb_embeddings.astype('float32'))
print(f"FAISS index ready with {index.ntotal} vectors.")

Loading CodeBERT for semantic retrieval...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Encoding training knowledge base (buggy methods)...


Indexing KB:   0%|          | 0/1637 [00:00<?, ?it/s]

FAISS index ready with 52364 vectors.


In [37]:
# ==========================================
# CELL 14: QWEN 1.5B BATCHED GENERATION & EVALUATION
# ==========================================
# Evict any lingering T5 models from previous cells to free up VRAM for Qwen
print("Status: Clearing VRAM from previous models...", flush=True)
if 'model_A' in globals(): del model_A
if 'model_B' in globals(): del model_B
if 'trainer_A' in globals(): del trainer_A
if 'trainer_B' in globals(): del trainer_B

gc.collect()
torch.cuda.empty_cache()

print("\nStatus: Initializing Qwen2.5-Coder-1.5B-Instruct...", flush=True)
qwen_checkpoint = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_checkpoint)

# Critical for Batched Generation: Use Left Padding
qwen_tokenizer.padding_side = "left"
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

# Force model to the primary device to avoid device-map offloading issues
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_checkpoint, 
    torch_dtype=torch.float16
).to(device) # Changed from device_map="auto" to explicit .to(device)

qwen_model.eval()
print(f"Status: Qwen model successfully loaded on {device}.", flush=True)

def run_qwen_eval(dataset, use_rag=False, k=3, batch_size=8):
    """
    Runs BATCHED evaluation for Qwen on the FULL dataset.
    Optimized for Quadro RTX 6000.
    """
    predictions = []
    references = dataset["fixed"]
    
    query_embeddings = None
    if use_rag:
        print(f"Status: Encoding {len(dataset)} test queries for RAG...", flush=True)
        query_embeddings = encode_code(dataset["buggy"], desc="Encoding Test Queries")

    print(f"Status: Starting BATCHED generation (BS={batch_size}) for {'RAG' if use_rag else 'Zero-Shot'}...", flush=True)
    
    for i in tqdm(range(0, len(dataset), batch_size), desc=f"Qwen ({'RAG' if use_rag else 'ZS'})"):
        batch_end = min(i + batch_size, len(dataset))
        batch_buggy = dataset["buggy"][i:batch_end]
        
        prompts = []
        for idx_in_batch, buggy_code in enumerate(batch_buggy):
            curr_idx = i + idx_in_batch
            if use_rag:
                query_emb = query_embeddings[curr_idx : curr_idx + 1]
                _, indices = index.search(query_emb.astype('float32'), k)
                
                examples_str = ""
                for rank, kb_idx in enumerate(indices[0]):
                    examples_str += f"### Example {rank+1}\nBuggy:\n{train_buggy[int(kb_idx)]}\nFixed:\n{train_fixed[int(kb_idx)]}\n\n"
                
                prompt = (
                    "You are an expert Java developer. Fix the bug in the Java method below.\n"
                    "Use the following similar examples for reference:\n\n"
                    f"{examples_str}"
                    "### Now fix this method\n"
                    f"Buggy:\n{buggy_code}\n\n"
                    "Fixed:"
                )
            else:
                prompt = f"You are an expert Java developer. Fix the bug in the following Java method.\n\nBuggy:\n{buggy_code}\n\nFixed:"
            
            prompts.append(prompt)

        # Ensure tokens are moved to the correct device
        inputs = qwen_tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)
        
        with torch.no_grad():
            outputs = qwen_model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=256, 
                do_sample=False, 
                pad_token_id=qwen_tokenizer.eos_token_id
            )
        
        # Extract predictions for the batch
        input_length = inputs.input_ids.shape[1]
        for output_ids in outputs:
            # Decode only the new tokens
            pred_text = qwen_tokenizer.decode(output_ids[input_length:], skip_special_tokens=True)
            # Standard cleanup for Qwen-Coder style output
            pred = pred_text.strip().split("###")[0].strip()
            predictions.append(pred)
    
    print(f"Status: Calculation metrics for {len(predictions)} samples...", flush=True)
    exact_matches = sum(1 for p, r in zip(predictions, references) if p.strip() == r.strip())
    em_score = (exact_matches / len(references)) * 100
    
    refs_for_bleu = [[r] for r in references]
    try:
        cb_results = calc_codebleu(references=refs_for_bleu, predictions=predictions, lang="java")
        codebleu_score = cb_results["codebleu"] * 100
    except Exception as e:
        codebleu_score = 0.0
        
    return {"Exact Match (%)": round(em_score, 4), "CodeBLEU": round(codebleu_score, 4)}

print(f"Starting Qwen benchmarks on the FULL test set (Batch Size 16)...", flush=True)
results_qwen_zs = run_qwen_eval(test_dataset, use_rag=False, batch_size=16)
results_qwen_rag = run_qwen_eval(test_dataset, use_rag=True, k=3, batch_size=16)

Status: Clearing VRAM from previous models...

Status: Initializing Qwen2.5-Coder-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Status: Qwen model successfully loaded on cuda.
Starting Qwen benchmarks on the FULL test set (Batch Size 16)...
Status: Starting BATCHED generation (BS=16) for Zero-Shot...


Qwen (ZS):   0%|          | 0/410 [00:00<?, ?it/s]

Status: Calculation metrics for 6545 samples...
Status: Encoding 6545 test queries for RAG...


Encoding Test Queries:   0%|          | 0/205 [00:00<?, ?it/s]

Status: Starting BATCHED generation (BS=16) for RAG...


Qwen (RAG):   0%|          | 0/410 [00:00<?, ?it/s]

Status: Calculation metrics for 6545 samples...


In [38]:
# ==========================================
# CELL 15: FINAL COMPARATIVE ANALYSIS
# ==========================================
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY (FULL TEST SET)")
print("="*60)
print(f"{'Configuration':<30} | {'Exact Match':<12} | {'CodeBLEU':<10}")
print("-" * 60)
print(f"{'Pipeline A (Pre-trained)':<30} | {results_A['Exact Match (%)']:>11}% | {results_A['CodeBLEU']:>10}")
print(f"{'Pipeline B (Scratch)':<30} | {results_B['Exact Match (%)']:>11}% | {results_B['CodeBLEU']:>10}")
print(f"{'Qwen 1.5B (Zero-Shot)':<30} | {results_qwen_zs['Exact Match (%)']:>11}% | {results_qwen_zs['CodeBLEU']:>10}")
print(f"{'Qwen 1.5B (RAG 3-Shot)':<30} | {results_qwen_rag['Exact Match (%)']:>11}% | {results_qwen_rag['CodeBLEU']:>10}")
print("="*60)

final_summary = {
    "Pipeline A": results_A,
    "Pipeline B": results_B,
    "Qwen Zero-Shot": results_qwen_zs,
    "Qwen RAG": results_qwen_rag
}

with open("final_results_summary.json", "w") as f:
    json.dump(final_summary, f, indent=4)

print("Results saved to final_results_summary.json. Project Complete!")


FINAL RESULTS SUMMARY (FULL TEST SET)
Configuration                  | Exact Match  | CodeBLEU  
------------------------------------------------------------
Pipeline A (Pre-trained)       |         0.0% |     30.726
Pipeline B (Scratch)           |      0.0458% |    51.9902
Qwen 1.5B (Zero-Shot)          |         0.0% |    42.8432
Qwen 1.5B (RAG 3-Shot)         |         0.0% |    44.5091
Results saved to final_results_summary.json. Project Complete!
